In [9]:


!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-groq
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q langchain_text_splitters

In [10]:
import os
os.environ["GROQ_API_KEY"]="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"

from google.colab import files
uploaded=files.upload()

Saving DivyaRaut_AIML_Resume.pdf to DivyaRaut_AIML_Resume.pdf


In [11]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("Divya_Raut_Resume.pdf")
documents=loader.load()

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
chunk_size=1500,
chunk_overlap=200
)

docs=splitter.split_documents(documents)

In [13]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings=HuggingFaceEmbeddings(
model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db=FAISS.from_documents(
docs,
embeddings
)

/tmp/ipykernel_7393/1571458420.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
from langchain_groq import ChatGroq

llm=ChatGroq(
model="llama-3.3-70b-versatile"
)

In [26]:
while True:

    query=input("Ask: ")

    if query=="exit":
        break

    docs=db.similarity_search(
        query,
        k=5
    )

    context="\n".join(
[d.page_content for d in docs]
)

    prompt = f"""
Answer ONLY using the provided context.

Rules:
- Give accurate answers only from context.
- Do not make up information.
- If asked for name, return only the name.
- If asked for skills, return all skills listed.
-Extract ONLY work or internship experience.
Rules:
- Include only roles/jobs/internships
- Include company name
- Include role title
- Include duration
- Exclude projects
- Exclude skills
- Exclude certifications

- If asked for projects, return project names exactly.
- For list-type questions, use bullet points.
- If answer is not in context, say "Not found in document."

Context:
{context}

Question:
{query}
"""

    response=llm.invoke(prompt)

    print("\nAnswer:")
    print(response.content)
    print()

Ask: what is the experience

Answer:
* Cybersecurity Intern, Sunbeam Institute of Information Technology, Pune [Dec-2025 – Jan 2026]

Ask: what is name of candidate

Answer:
Divya Raut

Ask: what is the skills

Answer:
* Python (Pandas, NumPy, Scikit-learn)
* Streamlit
* Generative AI (GPT-4, ChatGPT, LLaMA, Gemini, RAG)
* Web Development (JS, Express, HTML, React)
* Professional Skills (Stakeholder Management, Leadership, Problem Solving)
* ML & DL (Regression, Classification, Clustering, LLMs)
* Vector Stores (FAISS, ChromaDB, Pinecone)
* Visualization (Matplotlib, Seaborn)
* Cloud - AWS EC2, EBS, RDS

Ask: exit
